# Case Study: Kit End Date Prediction

Goal: predict `kit_end` for each sales order in the inference set.

## Setup

In [1]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## Load data

In [2]:
train = pd.read_parquet('/Users/anikaachary/Desktop/Dell Data Science Case Study - March 2026/Data/train.parquet')
inference = pd.read_parquet('/Users/anikaachary/Desktop/Dell Data Science Case Study - March 2026/Data/inference.parquet')

train.head() # includes kit_end - what we are trying to predict

,sales_order_id,buid,site,orig_wo_id,tie_num,mcid,order_amt,quantity_produced,line_qty,cfs_flag,is_cfi,alt_type_desc,lob_desc,family_parent_desc,family_desc,download_date,kit_start,kit_end,est_ship_date
0,1025548771,11,KHO,22067915245,1,3JU,42935.31840,8,8,0,1,NSYJLWFYJI XTQZYNTSX LWTZU UGZ,UTBJWJILJ,LJSYWD,LJSYWD HXU,2025-10-15,2025-10-16,2025-10-17,2025-10-31
1,1027927868,11,KHO,22095717945,1,KHO,33128.74827,2,2,0,1,NSYJLWFYJI XTQZYNTSX LWTZU UGZ,UTBJWJILJ,HMFRUNTS,HMFRUNTS AXFSWS,2025-12-11,2025-12-12,2025-12-14,2026-03-16
2,1029939204,11,KHO,22112237255,1,KHO,17238.22614,1,1,0,1,NSYJLWFYJI XTQZYNTSX LWTZU UGZ,UTBJWJILJ,HMFRUNTS,HMFRUNTS UJ,2026-01-29,2026-01-30,2026-02-03,2026-06-17
3,1023379512,11,KHO,22082951775,1,3JU,34306.97076,6,6,0,1,NSYJLWFYJI XTQZYNTSX LWTZU UGZ,UTBJWJILJ,RNZWF,RNZWF UJ,2025-11-19,2025-11-29,2025-11-30,2025-11-12
4,1017735495,707,KHO,21983065475,1,KHO,6261.03846,1,1,0,0,NSYJLWFYJI XTQZYNTSX LWTZU UGZ,UTBJWJILJ,MZWFHFS,MZWFHFS UJ,2025-04-01,2025-04-03,2025-04-04,2025-04-16


In [3]:
inference.head()

,sales_order_id,buid,site,orig_wo_id,tie_num,mcid,order_amt,quantity_produced,line_qty,cfs_flag,is_cfi,alt_type_desc,lob_desc,family_parent_desc,family_desc,download_date,kit_start,est_ship_date
0,1026073335,11,KHO,22117639095,1,KHO,52526.77737,6,6,0,1,NSYJLWFYJI XTQZYNTSX LWTZU UGZ,UTBJWJILJ,HMFRUNTS,HMFRUNTS IXX,2026-02-12,2026-02-13,2026-02-19
1,1027826085,11,KHO,22114646815,1,3JU,253966.42809,1,1,0,1,NSYJLWFYJI XTQZYNTSX LWTZU UGZ,UTBJWJILJ,UFQT AJWIJ,UFQT AJWIJ UJ,2026-02-04,2026-02-10,2026-04-03
2,1029739753,11,KHO,22116037515,1,KHO,46049.54133,8,8,0,1,NSYJLWFYJI XTQZYNTSX LWTZU UGZ,UTBJWJILJ,HTSHMT,HTSHMT UJ,2026-02-09,2026-02-26,2026-04-20
3,1030454098,11,KHO,22117289705,1,KHO,187435.69692,10,3,0,1,NSYJLWFYJI XTQZYNTSX LWTZU UGZ,XYTWFLJ ZSXYWZHYZWJI,HQJJP K710,HQJJP-K710,2026-02-12,2026-02-12,2026-05-20
4,1029526599,11,KHO,22110817985,1,3JU,24400.99320,8,8,0,1,NSYJLWFYJI XTQZYNTSX LWTZU UGZ,UTBJWJILJ,UFYWFX,UFYWFX TJRW,2026-01-27,2026-02-11,2026-04-27


## Feature Engineering

Note: I had assisstance from Claude.

The inference data already has `kit_start`. What's missing is `kit_end`. The prediction will be based off of how many days the kit takes, as the interval from `kit_start` to `kit_end` will be more useful.

In [4]:
train["kit_duration"] = (train["kit_end"] - train["kit_start"]).dt.days 
# I checked the above statement with Claude - the reasoning is to turn dates into days.
# subtracting two date columns gives a time gap (e.g. "2 days 00:00:00")
# .dt.days pulls out just the whole number of days, e.g. 2

train = train[train["kit_duration"] >= 0] # drop negative durations. 

In [5]:
train.columns

Index(['sales_order_id', 'buid', 'site', 'orig_wo_id', 'tie_num', 'mcid',
       'order_amt', 'quantity_produced', 'line_qty', 'cfs_flag', 'is_cfi',
       'alt_type_desc', 'lob_desc', 'family_parent_desc', 'family_desc',
       'download_date', 'kit_start', 'kit_end', 'est_ship_date',
       'kit_duration'],
      dtype='object')

I initially separated cols as numerical, categorical, and date/time for a RandomForest classifier. However, Ridge regression needs weekday/flag columns as categories 

In [6]:
# real measurements
NUMS = ["order_amt", "quantity_produced", "line_qty", "days_to_kit_start", "days_to_ship"]
# labels: value is name/code
CATS = ["buid", "mcid", "tie_num", "alt_type_desc", "family_desc",
        "cfs_flag", "is_cfi", "download_dow", "kit_start_dow"]

Originally, I did not have `make_features` as a function - Claude suggested I make it automatic to use when validating the `inference` data. This saves a huge step in the automation process - I did not want to manually do the same thing for the `inference` cols.

In [7]:
def make_features(df):
    df = df.copy() # use a copy so original df isn't modified
    # something I did not initially think about was converting text and date to 
    # numbers - these for loops take care of that.
    for c in ["download_date", "kit_start", "est_ship_date"]:
        # make sure date cols are real: coerce turns bad values --> NaT (missing date)
        df[c] = pd.to_datetime(df[c], errors="coerce") 
    for c in ["quantity_produced", "line_qty"]:
        df[c] = pd.to_numeric(df[c], errors="coerce") # convert times from text --> numbers

    df["order_amt"] = np.log1p(df["order_amt"].clip(lower=0)) # floors negatives at 0 first
    # np.log1p(order_amt): ceiling for huge orders

    # new features that are easier for readability built from dates
    df["days_to_kit_start"] = (df["kit_start"] - df["download_date"]).dt.days # kit duration
    df["days_to_ship"] = (df["est_ship_date"] - df["kit_start"]).dt.days
    df["download_dow"] = df["download_date"].dt.dayofweek
    df["kit_start_dow"] = df["kit_start"].dt.dayofweek
    
    df[CATS] = df[CATS].astype(str) # turn every category col into text --> encoder reads as label
    return df[NUMS + CATS]

In [8]:
train = train.drop_duplicates().copy()
train["kit_start"] = pd.to_datetime(train["kit_start"])
train["kit_end"]   = pd.to_datetime(train["kit_end"])
train["kit_duration"] = (train["kit_end"] - train["kit_start"]).dt.days # days

# Claude suggested this edit - dropping durations below 0 or 30 days. It's better to 
# have the training data on a focused range.
# negatives are impossible: data errors.
# >30 dats is extreme.
train = (train[train["kit_duration"].between(0, 30)]
         .sort_values("kit_start").reset_index(drop=True))

X = make_features(train) # inputs
y = train["kit_duration"] # this is what we are prediciting.

Originally, I tried to make use of sklearn's `get_dummies()` function to manually encode categorical and datetime variables for a random forest classifier - however, this process was tedious.

In [9]:
preprocess = ColumnTransformer([ # columnTransformer allows for selective encoding.
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUMS),
    # simple imputer labels cols 0/1 per category

    # Claude suggested a way to handle never seen before categories with unknown="ignore"
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATS), 
])

model = Pipeline([
    ("prep", preprocess),
    ("ridge", Ridge(alpha=1.0)), # fit ridge
])

# this pipeline learns from training data and then applied to inference 

In [10]:
cutoff = train["kit_start"].quantile(0.8) # trains on oldest 80% of orders; newest 20%
tr = train["kit_start"] < cutoff # true: training row, false: test row

# Claude suggested this, as well as adding % of orders predicted on the exact day and
# % predicted within ±1 day
def report(name, pred, actual):
    err = np.abs(pred - actual) # days off for each order
    # MAE = avg days off; lower = better
    print(f"{name:<20} MAE={err.mean():.2f} days | exact={np.mean(err==0):.1%} | within 1 day={np.mean(err<=1):.1%}")

model.fit(X[tr], y[tr]) # train on older 80% only
pred = np.clip(np.round(model.predict(X[~tr])), 0, None) # predict the newer 20%
base = np.full(len(pred), y[tr].median()) # round to whole days, no negatives

report("Baseline (median)", base, y[~tr]) # baseline: typical duration for order
report("Ridge", pred, y[~tr])

Baseline (median)    MAE=0.93 days | exact=45.1% | within 1 day=78.7%
Ridge                MAE=0.84 days | exact=44.4% | within 1 day=83.5%


In [11]:
model.set_params(ridge__alpha=1.0).fit(X[tr], y[tr]) #re-fit 

# names of every col after encoding
# = "is this order in family ..." 1/0 = yes/no
names = model.named_steps["prep"].get_feature_names_out()
# coef = days added + or removed - when col is 1
coefs = pd.Series(model.named_steps["ridge"].coef_, index=names).sort_values()

coefs.head(10)
coefs.tail(10)

cat__family_desc_LRFXYJWKZQQ JC5000     1.740634
cat__family_desc_XYWTSLGTB IXX          1.845313
cat__family_desc_QJTS IXX               1.860229
cat__family_desc_XYWTSLGTB TJRW         1.938442
cat__family_desc_FGNQJSJ IXX            2.351983
cat__family_desc_RTXX IXX               2.483436
cat__family_desc_RNSJWFQBJQQX AXFSWS    2.559921
cat__family_desc_YNRUXTS IXX            3.204718
cat__family_desc_UTBJWKQJC W650 WHP     4.654193
cat__family_desc_PJJHMN IXX             7.100893
dtype: float64

In [12]:
model.fit(X, y) # retrain on all training orders

X_inf = make_features(inference)
# predicted days per order: rounded to whole days, no negatives, stored as integers
dur = np.clip(np.round(model.predict(X_inf)), 0, None).astype(int)
kit_end_pred = pd.to_datetime(inference["kit_start"]) + pd.to_timedelta(dur, unit="D")

np.save("predictions.npy", kit_end_pred.values.astype("datetime64[D]"))
# predicted end date = start date + predicted days (e.g. Feb 11 + 2 days = Feb 13)

print(pd.Series(dur).value_counts().sort_index()) # how mant orders predicted at each duration

1    6879
2    4836
3     326
4      28
Name: count, dtype: int64
